# RegTech Demo Setup
This notebook provisions all Snowflake objects required for the RegTech AI Platform demo.
Run each cell in order.

**Pipeline overview:**
1. Database & schema
2. Create tables
3. **Fetch real US banking regulations** from eCFR API -> stage -> load text
4. **Extract structured requirements** using `COMPLETE()` + JSON parsing
5. Load balance sheet metrics
6. Train ML anomaly detection -> **dynamically explain with `COMPLETE()`**
7. Compute QoQ variance
8. **Run agent-powered audit pipeline** (`audit_pipeline.py`)
9. Create Cortex Search, Semantic View, and Cortex Agent

## Step 1: Database & Schema Setup

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE COMPUTE_WH;

CREATE DATABASE IF NOT EXISTS REGTECH_DEMO_DB;
USE DATABASE REGTECH_DEMO_DB;

CREATE SCHEMA IF NOT EXISTS REGULATORY_REPORTING;
USE SCHEMA REGULATORY_REPORTING;

## Step 2: Create Tables

In [ ]:
-- Regulatory documents store
CREATE OR REPLACE TABLE REGULATORY_DOCUMENTS (
    DOC_ID VARCHAR(50) PRIMARY KEY, FRAMEWORK VARCHAR(50), TITLE VARCHAR(500),
    CHAPTER VARCHAR(200), VERSION VARCHAR(10), EFFECTIVE_DATE DATE,
    STATUS VARCHAR(20), RAW_TEXT TEXT, FILE_PATH VARCHAR(500),
    PARSE_MODE VARCHAR(20), PAGE_COUNT INTEGER, RAW_METADATA VARIANT,
    LOADED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE EXTRACTED_REQUIREMENTS (
    REQ_ID VARCHAR(50) PRIMARY KEY, DOC_ID VARCHAR(50), CATEGORY VARCHAR(100),
    REQUIREMENT TEXT, THRESHOLD VARCHAR(200), SEVERITY VARCHAR(20),
    IMPACT_AREA VARCHAR(200), EXTRACTED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE BALANCE_SHEET_METRICS (
    METRIC_ID VARCHAR(30) PRIMARY KEY, QUARTER VARCHAR(10), QUARTER_DATE DATE,
    BUSINESS_LINE VARCHAR(50), CET1_RATIO FLOAT, TIER1_RATIO FLOAT,
    RWA_TOTAL_B FLOAT, LCR_RATIO FLOAT, LEVERAGE_RATIO FLOAT,
    NSFR_RATIO FLOAT, CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE ANOMALY_FLAGS (
    FLAG_ID VARCHAR(30) PRIMARY KEY, QUARTER VARCHAR(10), QUARTER_DATE DATE,
    BUSINESS_LINE VARCHAR(50), RWA_ACTUAL_B FLOAT, RWA_EXPECTED_B FLOAT,
    IS_ANOMALY BOOLEAN, ANOMALY_SCORE FLOAT, ANOMALY_REASON TEXT,
    DETECTED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE VARIANCE_ANALYSIS (
    VAR_ID VARCHAR(30) PRIMARY KEY, QUARTER VARCHAR(10), BUSINESS_LINE VARCHAR(50),
    METRIC VARCHAR(50), CURRENT_VALUE FLOAT, PRIOR_VALUE FLOAT,
    QOQ_CHANGE_PCT FLOAT, STATUS VARCHAR(20)
);

CREATE OR REPLACE TABLE AUDIT_FINDINGS (
    FINDING_ID VARCHAR(20) PRIMARY KEY, PIPELINE_NAME VARCHAR(100),
    SEVERITY VARCHAR(20), CATEGORY VARCHAR(50), DESCRIPTION TEXT,
    AFFECTED_TABLE VARCHAR(100), OLD_LOGIC TEXT, SUGGESTED_FIX TEXT,
    REGULATION_REF VARCHAR(100), DETECTED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE AUDIT_RUN_LOG (
    RUN_ID VARCHAR(50) PRIMARY KEY, RUN_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    STATUS VARCHAR(20), PIPELINES_SCANNED INTEGER, FINDINGS_COUNT INTEGER,
    CRITICAL_COUNT INTEGER, HIGH_COUNT INTEGER, MEDIUM_COUNT INTEGER,
    DURATION_SECONDS FLOAT
);

## Step 3: Fetch Regulations from eCFR API & Upload to Stage

1. Creates a file format + internal stage
2. Run `fetch_regulations.py` locally to download from [eCFR](https://www.ecfr.gov)
3. Upload text files via `snow stage copy` (locally)
4. Load text into `REGULATORY_DOCUMENTS`

**Sections:** Title 12 Part 3 Subparts A-F (Capital Adequacy), Part 50 Subpart B (LCR)

In [ ]:
CREATE OR REPLACE FILE FORMAT UTF8_TEXT_FORMAT
    TYPE = 'CSV'
    FIELD_DELIMITER = NONE
    RECORD_DELIMITER = '\n'
    SKIP_HEADER = 0
    FIELD_OPTIONALLY_ENCLOSED_BY = NONE
    ESCAPE = NONE;

CREATE OR REPLACE STAGE REGULATION_DOCS_STAGE
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'eCFR Title 12 banking regulation text files';

### 3a: Fetch regulations & upload to stage (run locally)

The following commands **must run on your local machine** (not in Snowsight):

```bash
cd regtech_demo

# Option A: run the helper script (does steps 1 & 2)
bash local_setup.sh

# Option B: run manually
python3 fetch_regulations.py
snow stage copy './regulations/*.txt' @REGULATION_DOCS_STAGE/regulations/ \\
  --connection MY_DEMO --overwrite --auto-compress false
```

Then run the next two cells to refresh the directory listing and verify the upload.

In [ ]:
ALTER STAGE REGULATION_DOCS_STAGE REFRESH;
SELECT RELATIVE_PATH, SIZE, LAST_MODIFIED FROM DIRECTORY(@REGULATION_DOCS_STAGE);

In [ ]:
-- Verify files landed (expect 7 .txt files)
SELECT COUNT(*) AS FILE_COUNT FROM DIRECTORY(@REGULATION_DOCS_STAGE)
WHERE RELATIVE_PATH LIKE '%.txt';

### 3b: Load regulation text from stage into `REGULATORY_DOCUMENTS`

In [ ]:
INSERT INTO REGULATORY_DOCUMENTS (DOC_ID, FRAMEWORK, TITLE, CHAPTER, VERSION, EFFECTIVE_DATE, STATUS, RAW_TEXT, FILE_PATH, PARSE_MODE)
WITH staged AS (
    SELECT
        METADATA$FILENAME AS FILE_PATH,
        METADATA$FILE_ROW_NUMBER AS ROW_NUM,
        $1 AS LINE
    FROM @REGULATION_DOCS_STAGE/regulations/ (FILE_FORMAT => 'UTF8_TEXT_FORMAT')
),
file_text AS (
    SELECT FILE_PATH,
        LISTAGG(LINE, '\n') WITHIN GROUP (ORDER BY ROW_NUM) AS RAW_TEXT
    FROM staged
    GROUP BY FILE_PATH
)
SELECT
    'DOC-ECFR-' || ROW_NUMBER() OVER (ORDER BY f.FILE_PATH),
    'US Federal Regulation (12 CFR)',
    REPLACE(REPLACE(SPLIT_PART(f.FILE_PATH, '/', -1), '.txt', ''), '_', ' '),
    SPLIT_PART(f.FILE_PATH, '/', -1),
    '2025', CURRENT_DATE(), 'Active',
    f.RAW_TEXT,
    f.FILE_PATH, 'TEXT'
FROM file_text f;

In [ ]:
SELECT DOC_ID, TITLE, PARSE_MODE, PAGE_COUNT, LENGTH(RAW_TEXT) AS TEXT_LEN
FROM REGULATORY_DOCUMENTS ORDER BY DOC_ID;

## Step 4: Extract Requirements with `COMPLETE()`

Uses `SNOWFLAKE.CORTEX.COMPLETE()` to extract structured regulatory requirements from each document,
then parses the JSON array and flattens into rows.

In [ ]:
INSERT INTO EXTRACTED_REQUIREMENTS (REQ_ID, DOC_ID, CATEGORY, REQUIREMENT, THRESHOLD, SEVERITY, IMPACT_AREA)
WITH extracted AS (
    SELECT DOC_ID,
        SNOWFLAKE.CORTEX.COMPLETE('claude-sonnet-4-6',
            'You are a banking regulation analyst. Extract key regulatory requirements from this text.\n'
            || 'Return ONLY a JSON object with exactly this structure (no markdown, no explanation):\n'
            || '{"category": "<Capital Adequacy|Market Risk|Liquidity Risk|Leverage|Operational Risk|Credit Risk>",\n'
            || ' "requirements": [\n'
            || '   {"requirement_text": "...", "threshold": "...", "severity": "Critical|High|Medium", "impact_area": "..."}\n'
            || ' ]}\n\n'
            || 'Extract up to 5 of the most important requirements. Text:\n\n'
            || LEFT(RAW_TEXT, 15000)
        ) AS llm_response
    FROM REGULATORY_DOCUMENTS
),
parsed AS (
    SELECT DOC_ID,
        TRY_PARSE_JSON(llm_response) AS parsed_json
    FROM extracted
    WHERE TRY_PARSE_JSON(llm_response) IS NOT NULL
)
SELECT
    'REQ-' || LPAD(ROW_NUMBER() OVER (ORDER BY p.DOC_ID, r.INDEX)::VARCHAR, 3, '0'),
    p.DOC_ID,
    p.parsed_json:category::VARCHAR,
    r.VALUE:requirement_text::VARCHAR,
    r.VALUE:threshold::VARCHAR,
    COALESCE(r.VALUE:severity::VARCHAR, 'Medium'),
    r.VALUE:impact_area::VARCHAR
FROM parsed p, LATERAL FLATTEN(INPUT => p.parsed_json:requirements) r;

In [ ]:
SELECT REQ_ID, DOC_ID, CATEGORY, SEVERITY, LEFT(REQUIREMENT, 120) AS PREVIEW
FROM EXTRACTED_REQUIREMENTS
ORDER BY CASE SEVERITY WHEN 'Critical' THEN 1 WHEN 'High' THEN 2 ELSE 3 END, REQ_ID;

## Step 5: Load Balance Sheet Metrics (8 quarters x 3 business lines)

In [ ]:
INSERT INTO BALANCE_SHEET_METRICS
(METRIC_ID, QUARTER, QUARTER_DATE, BUSINESS_LINE, CET1_RATIO, TIER1_RATIO, RWA_TOTAL_B, LCR_RATIO, LEVERAGE_RATIO, NSFR_RATIO) VALUES
('BSM-IS-Q12024','Q1 2024','2024-03-31','Institutional Securities',13.8,15.2,168.4,128.5,5.8,112.3),
('BSM-IS-Q22024','Q2 2024','2024-06-30','Institutional Securities',14.1,15.6,172.1,131.2,5.9,114.7),
('BSM-IS-Q32024','Q3 2024','2024-09-30','Institutional Securities',14.3,15.8,176.8,133.8,6.0,116.1),
('BSM-IS-Q42024','Q4 2024','2024-12-31','Institutional Securities',14.6,16.1,184.7,135.4,6.1,118.3),
('BSM-IS-Q12025','Q1 2025','2025-03-31','Institutional Securities',14.4,15.9,182.3,134.1,6.0,117.2),
('BSM-IS-Q22025','Q2 2025','2025-06-30','Institutional Securities',14.2,15.7,179.5,132.7,5.9,115.8),
('BSM-IS-Q32025','Q3 2025','2025-09-30','Institutional Securities',14.5,16.0,185.2,136.9,6.1,119.4),
('BSM-IS-Q42025','Q4 2025','2025-12-31','Institutional Securities',12.1,13.8,243.6,118.3,5.2,103.7),
('BSM-WM-Q12024','Q1 2024','2024-03-31','Wealth Management',16.2,17.8,82.4,142.3,7.1,124.6),
('BSM-WM-Q22024','Q2 2024','2024-06-30','Wealth Management',16.5,18.1,84.7,144.8,7.2,126.3),
('BSM-WM-Q32024','Q3 2024','2024-09-30','Wealth Management',16.8,18.4,86.2,146.1,7.3,127.8),
('BSM-WM-Q42024','Q4 2024','2024-12-31','Wealth Management',17.1,18.7,88.5,148.3,7.4,129.5),
('BSM-WM-Q12025','Q1 2025','2025-03-31','Wealth Management',17.0,18.6,87.8,147.6,7.4,128.9),
('BSM-WM-Q22025','Q2 2025','2025-06-30','Wealth Management',17.3,18.9,89.4,149.2,7.5,130.7),
('BSM-WM-Q32025','Q3 2025','2025-09-30','Wealth Management',17.5,19.1,91.2,151.4,7.6,132.1),
('BSM-WM-Q42025','Q4 2025','2025-12-31','Wealth Management',17.6,19.2,92.8,152.7,7.7,133.4),
('BSM-IM-Q12024','Q1 2024','2024-03-31','Investment Management',18.4,20.1,41.3,156.7,8.2,138.4),
('BSM-IM-Q22024','Q2 2024','2024-06-30','Investment Management',18.7,20.4,42.1,158.3,8.3,140.1),
('BSM-IM-Q32024','Q3 2024','2024-09-30','Investment Management',18.9,20.6,43.0,159.8,8.4,141.6),
('BSM-IM-Q42024','Q4 2024','2024-12-31','Investment Management',19.2,20.9,44.2,161.5,8.5,143.2),
('BSM-IM-Q12025','Q1 2025','2025-03-31','Investment Management',19.1,20.8,43.8,160.9,8.4,142.7),
('BSM-IM-Q22025','Q2 2025','2025-06-30','Investment Management',19.4,21.1,44.7,162.4,8.6,144.3),
('BSM-IM-Q32025','Q3 2025','2025-09-30','Investment Management',19.6,21.3,45.6,164.1,8.7,145.8),
('BSM-IM-Q42025','Q4 2025','2025-12-31','Investment Management',19.8,21.5,46.4,165.3,8.8,147.2);

## Step 6: Train ML Anomaly Detection & Dynamic Explanations

1. Train `RWA_ANOMALY_MODEL` on Q1 2024-Q3 2025
2. Score all quarters
3. **`SNOWFLAKE.CORTEX.COMPLETE()` generates anomaly explanations dynamically**

In [ ]:
CREATE OR REPLACE VIEW RWA_TRAINING_DATA AS
SELECT QUARTER_DATE AS TS, RWA_TOTAL_B AS Y, BUSINESS_LINE, FALSE AS IS_ANOMALY
FROM BALANCE_SHEET_METRICS WHERE QUARTER != 'Q4 2025';

CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION RWA_ANOMALY_MODEL(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'RWA_TRAINING_DATA'),
    SERIES_COLNAME => 'BUSINESS_LINE',
    TIMESTAMP_COLNAME => 'TS',
    TARGET_COLNAME => 'Y',
    LABEL_COLNAME => 'IS_ANOMALY'
);

In [ ]:
-- Score only Q4 2025 (must be after training window)
CREATE OR REPLACE VIEW RWA_SCORING_DATA AS
SELECT QUARTER_DATE, RWA_TOTAL_B, BUSINESS_LINE
FROM BALANCE_SHEET_METRICS WHERE QUARTER = 'Q4 2025';

CREATE OR REPLACE TABLE ANOMALY_DETECTION_RAW AS
SELECT * FROM TABLE(RWA_ANOMALY_MODEL!DETECT_ANOMALIES(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'RWA_SCORING_DATA'),
    SERIES_COLNAME => 'BUSINESS_LINE',
    TIMESTAMP_COLNAME => 'QUARTER_DATE',
    TARGET_COLNAME => 'RWA_TOTAL_B',
    CONFIG_OBJECT => {'prediction_interval': 0.95}
));

### 6c: Dynamic anomaly explanations via `COMPLETE()`

In [ ]:
INSERT INTO ANOMALY_FLAGS
WITH scored AS (
    SELECT
        'AF-' || ROW_NUMBER() OVER (ORDER BY r.SERIES, r.TS) AS FLAG_ID,
        m.QUARTER, r.TS AS QUARTER_DATE, r.SERIES AS BUSINESS_LINE,
        r.Y AS RWA_ACTUAL_B, r.FORECAST AS RWA_EXPECTED_B,
        r.IS_ANOMALY, r.PERCENTILE AS ANOMALY_SCORE
    FROM ANOMALY_DETECTION_RAW r
    JOIN BALANCE_SHEET_METRICS m ON r.TS = m.QUARTER_DATE AND r.SERIES = m.BUSINESS_LINE
)
SELECT s.FLAG_ID, s.QUARTER, s.QUARTER_DATE, s.BUSINESS_LINE,
    s.RWA_ACTUAL_B, s.RWA_EXPECTED_B, s.IS_ANOMALY, s.ANOMALY_SCORE,
    CASE WHEN s.IS_ANOMALY THEN
        SNOWFLAKE.CORTEX.COMPLETE('claude-sonnet-4-6',
            'You are a regulatory risk analyst. Generate a concise 2-3 sentence explanation for this RWA anomaly. '
            || 'Business Line: ' || s.BUSINESS_LINE
            || '. Quarter: ' || s.QUARTER
            || '. Actual RWA: $' || ROUND(s.RWA_ACTUAL_B, 1)::VARCHAR || 'B'
            || '. Expected RWA: $' || ROUND(s.RWA_EXPECTED_B, 1)::VARCHAR || 'B'
            || '. QoQ Change: ' || ROUND((s.RWA_ACTUAL_B - s.RWA_EXPECTED_B) / NULLIF(s.RWA_EXPECTED_B, 0) * 100, 1)::VARCHAR || '%'
            || '. Anomaly Score: ' || ROUND(s.ANOMALY_SCORE, 3)::VARCHAR
            || '. Reference Basel III/IV regulations (FRTB, SA-CCR, output floor). Be specific about root causes.'
        )
    ELSE NULL END AS ANOMALY_REASON,
    CURRENT_TIMESTAMP() AS DETECTED_AT
FROM scored s;

## Step 7: Compute Variance Analysis

In [ ]:
INSERT INTO VARIANCE_ANALYSIS
WITH ranked AS (
    SELECT QUARTER, BUSINESS_LINE, CET1_RATIO, TIER1_RATIO, RWA_TOTAL_B, LCR_RATIO,
        LAG(CET1_RATIO)  OVER (PARTITION BY BUSINESS_LINE ORDER BY QUARTER_DATE) AS PREV_CET1,
        LAG(RWA_TOTAL_B) OVER (PARTITION BY BUSINESS_LINE ORDER BY QUARTER_DATE) AS PREV_RWA,
        LAG(LCR_RATIO)   OVER (PARTITION BY BUSINESS_LINE ORDER BY QUARTER_DATE) AS PREV_LCR,
        ROW_NUMBER() OVER (PARTITION BY BUSINESS_LINE ORDER BY QUARTER_DATE DESC) AS RN
    FROM BALANCE_SHEET_METRICS
)
SELECT 'VAR-' || ROW_NUMBER() OVER (ORDER BY BUSINESS_LINE, QUARTER), QUARTER, BUSINESS_LINE,
    METRIC, CURRENT_VALUE, PRIOR_VALUE,
    ROUND((CURRENT_VALUE - PRIOR_VALUE) / NULLIF(PRIOR_VALUE, 0) * 100, 2),
    CASE WHEN ABS((CURRENT_VALUE - PRIOR_VALUE) / NULLIF(PRIOR_VALUE, 0)) > 0.15 THEN 'BREACH'
         WHEN ABS((CURRENT_VALUE - PRIOR_VALUE) / NULLIF(PRIOR_VALUE, 0)) > 0.05 THEN 'WARNING'
         ELSE 'NORMAL' END
FROM (
    SELECT QUARTER, BUSINESS_LINE, 'CET1_RATIO' AS METRIC, CET1_RATIO AS CURRENT_VALUE, PREV_CET1 AS PRIOR_VALUE FROM ranked WHERE RN = 1 AND PREV_CET1 IS NOT NULL
    UNION ALL
    SELECT QUARTER, BUSINESS_LINE, 'RWA_TOTAL_B', RWA_TOTAL_B, PREV_RWA FROM ranked WHERE RN = 1 AND PREV_RWA IS NOT NULL
    UNION ALL
    SELECT QUARTER, BUSINESS_LINE, 'LCR_RATIO', LCR_RATIO, PREV_LCR FROM ranked WHERE RN = 1 AND PREV_LCR IS NOT NULL
) t;

## Step 8: Run Agent-Powered Audit Pipeline (run locally)

`audit_pipeline.py` uses `cortex_code_agent_sdk` to analyze 8 SQL pipeline files against extracted requirements.

**Run on your local machine** (not in Snowsight):

```bash
cd regtech_demo
python3 audit_pipeline.py --connection MY_DEMO --force
```

| Pipeline | Intentional Issue |
|---|---|
| `market_risk_rwa.sql` | VaR instead of ES (FRTB) |
| `derivatives_ead.sql` | CEM instead of SA-CCR |
| `credit_rwa_irb.sql` | Missing 72.5% output floor |
| `cet1_deductions.sql` | Single threshold, not dual 17.65% |
| `lcr_hqla_buffer.sql` | RMBS as Level 2A, not 2B |
| `op_risk_capital.sql` | BIA instead of SMA |
| `mortgage_rwa.sql` | Flat 35% instead of LTV-based |
| `leverage_ratio.sql` | Missing written CDS exposure |

Then run the cells below to verify findings were written.

In [ ]:
-- Verify audit findings
SELECT FINDING_ID, PIPELINE_NAME, SEVERITY, CATEGORY,
       LEFT(DESCRIPTION, 100) AS PREVIEW, REGULATION_REF
FROM AUDIT_FINDINGS
ORDER BY CASE SEVERITY WHEN 'Critical' THEN 1 WHEN 'High' THEN 2 ELSE 3 END, FINDING_ID;

In [ ]:
SELECT * FROM AUDIT_RUN_LOG ORDER BY RUN_TIMESTAMP DESC LIMIT 5;

## Step 9: Create Cortex Search Service

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE REGULATORY_DOCS_SEARCH
ON RAW_TEXT
ATTRIBUTES FRAMEWORK, TITLE, CHAPTER, VERSION, STATUS
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 hour'
AS (
    SELECT DOC_ID, FRAMEWORK, TITLE, CHAPTER, VERSION, STATUS, RAW_TEXT
    FROM REGULATORY_DOCUMENTS
);

## Step 10: Create Semantic View

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW REGTECH_ANALYTICS
TABLES (
    balance_sheet AS BALANCE_SHEET_METRICS PRIMARY KEY (METRIC_ID),
    anomalies AS ANOMALY_FLAGS PRIMARY KEY (FLAG_ID),
    variance AS VARIANCE_ANALYSIS PRIMARY KEY (VAR_ID),
    audit AS AUDIT_FINDINGS PRIMARY KEY (FINDING_ID)
)
FACTS (
    balance_sheet.cet1 AS CET1_RATIO,
    balance_sheet.tier1 AS TIER1_RATIO,
    balance_sheet.rwa AS RWA_TOTAL_B,
    balance_sheet.lcr AS LCR_RATIO,
    balance_sheet.leverage AS LEVERAGE_RATIO,
    balance_sheet.nsfr AS NSFR_RATIO,
    anomalies.anomaly_score AS ANOMALY_SCORE,
    anomalies.rwa_actual AS RWA_ACTUAL_B,
    anomalies.rwa_expected AS RWA_EXPECTED_B,
    variance.qoq_change AS QOQ_CHANGE_PCT
)
DIMENSIONS (
    balance_sheet.quarter AS QUARTER,
    balance_sheet.business_line AS BUSINESS_LINE,
    anomalies.is_anomaly AS IS_ANOMALY,
    anomalies.anomaly_reason AS ANOMALY_REASON,
    audit.severity AS SEVERITY,
    audit.pipeline_name AS PIPELINE_NAME,
    audit.regulation_ref AS REGULATION_REF,
    variance.var_status AS STATUS
)
METRICS (
    balance_sheet.avg_cet1_ratio AS AVG(balance_sheet.CET1_RATIO),
    balance_sheet.total_rwa AS SUM(balance_sheet.RWA_TOTAL_B),
    anomalies.anomaly_count AS COUNT_IF(anomalies.IS_ANOMALY = TRUE),
    audit.critical_findings AS COUNT_IF(audit.SEVERITY = 'Critical')
);

## Step 11: Create Cortex Agent

In [ ]:
CREATE OR REPLACE AGENT REGTECH_ANALYTICS_AGENT
COMMENT = 'AI analyst for regulatory capital metrics, RWA anomaly detection, and Basel IV pipeline compliance.'
FROM SPECIFICATION $$
models:
  orchestration: claude-sonnet-4-6
instructions:
  response: "You are a regulatory technology analyst for a large bank. Answer questions about capital ratios, RWA trends, anomaly flags, audit findings, and regulatory compliance. Be precise with numbers and cite specific regulations."
  orchestration: "For quantitative questions about metrics, ratios, or trends use Analyst. For questions about regulatory text or compliance requirements use Search."
tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "Analyst"
      description: "Query balance sheet metrics, anomaly flags, variance analysis, and audit findings"
  - tool_spec:
      type: "cortex_search"
      name: "Search"
      description: "Search US banking regulations (12 CFR) for capital adequacy, liquidity, and market risk rules"
tool_resources:
  Analyst:
    execution_environment:
      type: warehouse
      warehouse: COMPUTE_WH
    semantic_view: "REGTECH_DEMO_DB.REGULATORY_REPORTING.REGTECH_ANALYTICS"
  Search:
    name: "REGTECH_DEMO_DB.REGULATORY_REPORTING.REGULATORY_DOCS_SEARCH"
    max_results: "5"
$$;

## Step 12: Verify Setup

In [ ]:
SELECT 'REGULATORY_DOCUMENTS' AS table_name, COUNT(*) AS row_count FROM REGULATORY_DOCUMENTS
UNION ALL SELECT 'EXTRACTED_REQUIREMENTS', COUNT(*) FROM EXTRACTED_REQUIREMENTS
UNION ALL SELECT 'BALANCE_SHEET_METRICS', COUNT(*) FROM BALANCE_SHEET_METRICS
UNION ALL SELECT 'ANOMALY_FLAGS', COUNT(*) FROM ANOMALY_FLAGS
UNION ALL SELECT 'VARIANCE_ANALYSIS', COUNT(*) FROM VARIANCE_ANALYSIS
UNION ALL SELECT 'AUDIT_FINDINGS', COUNT(*) FROM AUDIT_FINDINGS
UNION ALL SELECT 'AUDIT_RUN_LOG', COUNT(*) FROM AUDIT_RUN_LOG
ORDER BY table_name;

In [ ]:
-- Verify anomaly with dynamic AI-generated explanation
SELECT QUARTER, BUSINESS_LINE, RWA_ACTUAL_B, RWA_EXPECTED_B,
       IS_ANOMALY, ANOMALY_SCORE, ANOMALY_REASON
FROM ANOMALY_FLAGS WHERE IS_ANOMALY = TRUE
ORDER BY ANOMALY_SCORE DESC;

In [ ]:
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'REGTECH_DEMO_DB.REGULATORY_REPORTING.REGULATORY_DOCS_SEARCH',
    '{"query": "expected shortfall FRTB trading book", "columns": ["TITLE", "CHAPTER", "RAW_TEXT"], "limit": 2}'
);